# **Imports and defining vehicle's speeds**

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

# Load the Dataset
df = pd.read_csv('../Dataset/clean_data_v2.csv')

# Define vehicle speeds for expected time calculation (km/h)
vehicle_speeds = {
    'truck': 60,
    'pickup': 70,
    'van': 65,
    'car': 80,
    'motorcycle': 75
}

In [2]:


# ============================================
# 1. Distance-based Features
# ============================================

# Create distance buckets based on quantiles
distance_25 = df['distance_km'].quantile(0.33)
distance_50 = df['distance_km'].quantile(0.67)

def create_distance_bucket(distance):
    if distance <= distance_25:
        return 'short'
    elif distance <= distance_50:
        return 'medium'
    else:
        return 'long'

df['distance_bucket'] = df['distance_km'].apply(create_distance_bucket)

# Create binary long distance feature
df['is_long_distance'] = (df['distance_km'] > distance_50).astype(int)

# ============================================
# 2. Expected Time without Traffic
# ============================================

# Calculate expected time without traffic
def calculate_expected_time(row):
    speed = vehicle_speeds.get(row['vehicle_type'], 70)  # default 70 if vehicle not in dict
    return row['distance_km'] / speed

df['expected_time_no_traffic'] = df.apply(calculate_expected_time, axis=1)

# ============================================
# 3. Interaction Features
# ============================================

# Create traffic_weather_risk score
weather_risk_map = {
    'clear': 0,  # Assuming clear exists in full Dataset
    'clouds': 1,
    'mist': 2,
    'rain': 3,   # Assuming rain exists
    'snow': 4    # Assuming snow exists
}

traffic_risk_map = {
    'low': 0,
    'medium': 1,
    'high': 2
}

# Map weather and traffic to numeric risk values
df['weather_risk'] = df['weather'].map(weather_risk_map).fillna(1)  # default 1 for unknown weather
df['traffic_risk'] = df['traffic_level'].map(traffic_risk_map).fillna(0)  # default 0 for unknown

# Create combined risk score
df['traffic_weather_risk'] = df['weather_risk'] * (df['traffic_risk'] + 1)

# Create night-peak conflict feature
df['night_peak_conflict'] = (df['is_night'] & df['is_peak_hour']).astype(int)

# Drop intermediate columns
df = df.drop(['weather_risk', 'traffic_risk'], axis=1)

# ============================================
# 4. Cyclic Encoding for Temporal Features
# ============================================

# Cyclic encoding for order_hour (24-hour cycle)
df['order_hour_sin'] = np.sin(2 * np.pi * df['order_hour'] / 24)
df['order_hour_cos'] = np.cos(2 * np.pi * df['order_hour'] / 24)

# Cyclic encoding for month (12-month cycle)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# ============================================
# 5. Final Processing and Saving
# ============================================

# Ensure all columns are in correct order
final_columns = [
    'origin_city', 'destination_city', 'distance_km', 'distance_bucket',
    'is_long_distance', 'vehicle_type', 'order_date', 'order_hour',
    'order_hour_sin', 'order_hour_cos', 'weekday', 'weather', 'temperature',
    'traffic_level', 'delivery_time_hours', 'day_of_month', 'week_of_year',
    'month', 'month_sin', 'month_cos', 'is_weekend', 'is_peak_hour', 'is_night',
    'expected_time_no_traffic', 'traffic_weather_risk', 'night_peak_conflict'
]

# Reorder columns
df = df[final_columns]



Feature engineering complete!
Original shape: (69926, 26)
Features saved to 'feature_data_v2.csv'
New features created:
- distance_bucket (categorical)
- is_long_distance (binary)
- expected_time_no_traffic (numeric)
- traffic_weather_risk (numeric)
- night_peak_conflict (binary)
- order_hour_sin, order_hour_cos (cyclic)
- month_sin, month_cos (cyclic)


## **Save Dataset**

In [ ]:

# Save to new CSV file
df.to_csv('feature_data_v2.csv', index=False)

print("Feature engineering complete!")
print(f"Original shape: {df.shape}")
print(f"Features saved to 'feature_data_v2.csv'")
print(f"New features created:")
print("- distance_bucket (categorical)")
print("- is_long_distance (binary)")
print("- expected_time_no_traffic (numeric)")
print("- traffic_weather_risk (numeric)")
print("- night_peak_conflict (binary)")
print("- order_hour_sin, order_hour_cos (cyclic)")
print("- month_sin, month_cos (cyclic)")